# Food-101 experiment workflow

**NOT RUN — authored for review only.** Supporting definitions for Notebook02.
Inputs: Food101 config, Run All orchestration, unchanged Notebook01 source/evidence, official archive only during the preparation step.
Outputs: stage manifests, checkpoints, four probes, diagnostics and quality/cost report when explicitly called.
Runtime: imports seconds; data audit and preflight minutes; pilot hours (measured by preflight).
Resume: same config/RUN_ID, main notebook Run All. All long training runs use detached tmux and one run writer.
Code is imported only through tagged `ahr-export` cells. No experiment starts on import.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)


In [ ]:
from pathlib import Path
import copy, datetime, hashlib, json, os, re, shlex, subprocess, sys, time, traceback
import torch
from fmm_jepa.training import atomic_json, digest_file, definition_hashes, run_writer, write_csv
from fmm_jepa.food101_training import initialize_paired_models, training_protocol, run_model
from fmm_jepa.food101_data import prepare_food101, make_batch, EvaluationDataset, preprocessing_descriptor
from fmm_jepa.food101_evaluation import compute_diagnostics
from fmm_jepa.tiny_workflow import run_environment

FOOD_SCHEMA = "fmm-jepa-food-config/v1"

def identity_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def consumer_source_hash(root):
    document = json.loads((Path(root) / "notebooks/02_Food101_Main.ipynb").read_text())
    # Run-id controls and saved outputs may change between explicit stages.
    sources = ["".join(c["source"]) for c in document["cells"]
        if c["cell_type"] == "code" and c.get("id") != "runtime-controls"]
    return identity_hash(sources)

def read_config(root, path="configs/food101_pilot.yaml"):
    config = json.loads((Path(root) / path).read_text())
    if config["schema_version"] != FOOD_SCHEMA: raise ValueError("Unsupported Food-101 config")
    if config["probe"]["num_classes"] != 101: raise ValueError("Food-101 requires 101 classes")
    if config["evaluation"]["heldout_enabled"]: raise ValueError("This pilot reserves official test data; register a separate final evaluation")
    if config["dtype"] != "float32" or config["world_size"] != 1: raise ValueError("Registered scope is single-device FP32")
    patch = config["model"]["patch_size"]; size = config["image_size"]
    if size % patch or size // patch not in (8, 16, 32): raise ValueError("Use a registered 8, 16 or 32 square token grid")
    if config["device"] == "auto": config["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"
    config["effective_batch_size"] = config["training"]["microbatch"] * config["training"]["accumulation_steps"]
    torch.set_num_threads(config["cpu_threads"])
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    training_protocol(config)
    return config

def artifact_path(root, run_id):
    if not isinstance(run_id, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,95}", run_id):
        raise ValueError("RUN_ID must contain only letters, digits, underscores and hyphens, starting with a letter or digit")
    return Path(root) / "artifacts/notebook_02" / run_id

def predecessor_evidence(root, config):
    previous = Path(root) / "artifacts/notebook_01" / config["predecessor_run_id"]
    handoff = json.loads((previous / "handoff_01_to_02.json").read_text())
    review = json.loads((previous / "interpretation_review.json").read_text())
    if not handoff["engineering_ready"] or not handoff["ready_for_notebook_02"] or review["status"] != "PASS":
        raise RuntimeError("Notebook 01 handoff or independent review does not support this stage")
    current = definition_hashes(root)
    for name, expected in handoff["source_hashes"].items():
        if current.get(name) != expected: raise RuntimeError(f"Predecessor source changed: {name}")
    return {"run_id": handoff["run_id"], "handoff_sha256": digest_file(previous / "handoff_01_to_02.json"),
            "review_sha256": digest_file(previous / "interpretation_review.json"), "status": "PASS",
            "scope": "saved Notebook01 evidence and source subset checked; new Food runtime checks remain unexecuted"}

def open_run(root, run_id, config):
    artifact = artifact_path(root, run_id)
    artifact.mkdir(parents=True, exist_ok=True)
    with run_writer(artifact):
        path = artifact / "resolved_config.json"
        if path.exists():
            if json.loads(path.read_text()) != config: raise ValueError("Registered config changed: restore it or choose a new RUN_ID")
        else:
            # Refuse evidence stranded without the config that identifies it.
            if any(p.name != ".writer.lock" for p in artifact.iterdir()):
                raise RuntimeError("Existing artifacts have no resolved config; inspect before recovery")
            atomic_json(path, config)
        for name in ("figures", "features", "predictions"):
            (artifact / name).mkdir(exist_ok=True)
    return artifact

def set_status(artifact, stage, status, **details):
    atomic_json(Path(artifact) / "pilot_status.json", {"schema": "fmm-jepa-food-status/v1",
        "run_id": Path(artifact).name, "stage": stage, "status": status,
        "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), **details})

def prepare_evidence(root, artifact, config, allow_download=False):
    bundle = prepare_food101(root, config, allow_download=allow_download)
    for key in ("dataset_manifest", "split_manifest"):
        atomic_json(Path(artifact) / (key + ".json"), bundle[key])
    atomic_json(Path(artifact) / "duplicate_audit.json", bundle["audit"])
    return bundle

def stable_environment(config):
    value = run_environment(config)
    # Instantaneously available resources are recorded separately, not protocol identity.
    return {k: v for k, v in value.items() if k not in ("available_ram_bytes", "cuda_free_total_bytes")}

def register_food_protocol(root, artifact, config, bundle):
    predecessor = predecessor_evidence(root, config)
    models, counts, initialization = initialize_paired_models(config, artifact)
    value = {"schema": "fmm-jepa-food-protocol/v1", "run_id": Path(artifact).name,
        "config": config, "source_hashes": definition_hashes(root),
        "consumer_notebook_sha256": consumer_source_hash(root),
        "predecessor": predecessor, "dataset_fingerprint": bundle["dataset"].fingerprint,
        "split_sha256": bundle["split_manifest"]["split_sha256"],
        "initialization": initialization, "parameter_counts": counts,
        "training": training_protocol(config), "environment": stable_environment(config),
        "data_pairing": "identical sample IDs, occurrences, augmentation and mask streams for both architectures",
        "comparison": "same scaffold, equal successful updates/exposures; not parameter or compute matched",
        "checkpoint_selection": "final registered update budget; best JEPA loss is recovery-only",
        "probe_selection": "30 equal epochs by default; best development top1, earliest tie; no official test evaluation",
        "primary_contrast": "FMM trained minus its exact initialization development top1",
        "learning_rule": "both full budgets and four probes complete; primary paired image CI lower > 0; no registered final variance/rank flag",
        "diagnostic_caveat": "rank<2 and variance<=1e-12 are advisory flags, not universal representation-quality guarantees",
        "uncertainty": "one paired seed; image bootstrap excludes training-seed and development-selection uncertainty",
        "stop_rules": "invalid provenance/checkpoints stop; OOM stops that benchmark case; preflight failure blocks training; repeated nonfinite attempts stop",
        "scope": "one bounded Food101 comparison, no automatic multi-seed or ablation expansion"}
    path = Path(artifact) / "protocol.json"
    if path.exists() and json.loads(path.read_text()) != value:
        raise RuntimeError("Protocol, source, data, initialization or stable environment changed; exact resume refused")
    atomic_json(path, value)
    atomic_json(Path(artifact) / "parameter_counts.json", counts)
    atomic_json(Path(artifact) / "environment.json", run_environment(config))
    return models, initialization, value


In [ ]:
def train_pair(root,artifact,run_id,config,bundle,models,protocol):
    from fmm_jepa.food101_data import EvaluationDataset
    from fmm_jepa.food101_training import run_model
    from fmm_jepa.food101_evaluation import compute_diagnostics
    selected = list(bundle['diagnostic_ids'])
    labels = {bundle['dataset'].get(sid)['label'] for sid in selected}
    if len(labels) != config['probe']['num_classes']: raise ValueError('Diagnostic set must cover every class')
    atomic_json(Path(artifact)/'diagnostic_membership.json', {'split':'development','sample_ids':selected,
        'count':len(selected),'selection':'registered fixed development examples per class'})
    diagnostic_data=EvaluationDataset(bundle['dataset'],selected,config)
    diagnostics=[]
    def callback(model,kind,step):
        value=compute_diagnostics(model.target,diagnostic_data,config)
        value.update(model_id=kind,seed=config['seeds']['model_init'],phase='representation_diagnostic',split='development',step=step)
        diagnostics.append(value);atomic_json(Path(artifact)/'representation_diagnostics.json',diagnostics)
        return value
    trained={}
    for kind in ('vit','fmm'):
        print(f'{kind}: starting/resuming registered {config["training"]["updates"]}-update budget',flush=True)
        trained[kind]=run_model(kind,config,bundle,artifact,Path(root)/'checkpoints'/run_id/kind,protocol,model=models[kind],diagnostic_callback=callback)
        print(f'{kind}: {trained[kind]["resume"]}',flush=True)
    # Resume retains diagnostic history in authoritative checkpoints.
    diagnostics=[]
    for kind,value in trained.items():
        for row in value['diagnostics']:
            if isinstance(row,dict) and 'results' in row:row={**row['results'],'model_id':kind,'step':row.get('step')}
            diagnostics.append(row)
    metrics=[{**row,'run_id':run_id,'checkpoint_manifest':str(Path(v['checkpoint_dir'])/'manifest.json'),'state_reference':run_id+'/'+row['model_id']+'/successful_update_'+str(row['successful_updates'])} for v in trained.values() for row in v['metrics']]
    atomic_json(Path(artifact)/'representation_diagnostics.json',diagnostics)
    write_csv(Path(artifact)/'representation_diagnostics.csv',diagnostics)
    write_csv(Path(artifact)/'training_metrics.csv',metrics)
    traces={k:v['traces'] for k,v in trained.items()}
    # Compare successful data events independently of model-specific loss and wall time.
    paired_fields=('sample_ids','augmentation_ids','mask_ids','occurrences')
    def data_trace(rows):
        return [{key:row.get(key) for key in paired_fields} for row in rows]
    paired=data_trace(traces['vit'])==data_trace(traces['fmm'])
    atomic_json(Path(artifact)/'paired_training_summary.json',{'paired_trace_equal':paired,'results':{k:v['resume'] for k,v in trained.items()}})
    if not paired:raise AssertionError('Architectures did not consume paired data/mask traces')
    return trained,diagnostics

def evaluate_four(root,artifact,run_id,config,bundle,trained,initialization):
    import gc
    from fmm_jepa.geometry import clear_geometry_cache
    from fmm_jepa.food101_data import EvaluationDataset,preprocessing_descriptor
    from fmm_jepa.food101_evaluation import encoder_state_hash,extract_features,fit_linear_probe,compare_representations,evaluate_heldout
    probes={};features={};performance=[];encoders={}
    for kind in ('vit','fmm'):
        for control in ('untrained','trained'):
            model_id=kind+'_'+control
            clear_geometry_cache();gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
            encoder=copy.deepcopy(trained[kind]['model'].target).cpu()
            if control=='untrained':
                record=initialization['states'][kind]['teacher']
                if digest_file(record['path'])!=record['sha256']:raise ValueError('Initial teacher changed')
                encoder.load_state_dict(torch.load(record['path'],map_location='cpu',weights_only=True))
            encoders[model_id]=encoder
            identity={'dataset_revision':bundle['dataset'].fingerprint,
                      'checkpoint_hash':encoder_state_hash(encoder),'encoder_kind':kind,
                      'preprocessing':preprocessing_descriptor(config),'pooling':'mean_final_layernorm_patches',
                      'normalization':'encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization'}
            caches={}
            for split,ids in [('train',bundle['train_ids']),('development',bundle['dev_ids'])]:
                print(f'{model_id}: extracting {len(ids)} {split} features',flush=True)
                caches[split]=extract_features(encoder,EvaluationDataset(bundle['dataset'],ids,config),Path(artifact)/'features'/model_id,
                    {**identity,'sample_ids':list(ids)},batch_size=config['evaluation']['batch_size'],device=config['device'])
                feature_perf=caches[split]['performance']
                if caches[split]['reused']:
                    original=json.loads(Path(caches[split]['path']).with_suffix('.json').read_text())['performance']
                    feature_perf={**original,'historical_measurement':True,'current_invocation':'NOT_RUN_valid_cache_reused'}
                performance.append({**feature_perf,'model_id':model_id,'split':split,'checkpoint_hash':identity['checkpoint_hash'],'seed':config['seeds']['model_init']})
            features[model_id]=caches
            clear_geometry_cache();gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
            probes[model_id]=fit_linear_probe(caches['train'],caches['development'],Path(root)/'checkpoints'/run_id/'probes',config,model_id,encoder=encoder)
            value=probes[model_id]
            write_csv(Path(artifact)/'predictions'/(model_id+'_development.csv'),value['predictions'])
            atomic_json(Path(artifact)/(model_id+'_probe_summary.json'),value['summary'])
            probe_perf_path=Path(artifact)/(model_id+'_probe_performance.json')
            if value['summary']['resumed'] and value['performance'].get('batches_measured',0)==0 and probe_perf_path.exists():
                value['performance']={**json.loads(probe_perf_path.read_text()),'historical_measurement':True,'current_invocation':'NOT_RUN_completed_probe_reused'}
            else:atomic_json(probe_perf_path,value['performance'])
            performance.append({**value['performance'],'model_id':model_id,'split':'train_and_development','checkpoint_hash':identity['checkpoint_hash'],'seed':config['seeds']['probe']})
            print(f'{model_id}: development {value["summary"]["metrics"]}',flush=True)
            gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
    comparison=compare_representations(probes,config)
    atomic_json(Path(artifact)/'heldout_results.json', {'status':'NOT_RUN',
        'reason':'Official Food101 test reserved; no held-out features or predictions computed.'})
    write_csv(Path(artifact)/'probe_metrics.csv',[r for v in probes.values() for r in v['metrics']])
    atomic_json(Path(artifact)/'comparison_summary.json',comparison)
    return probes,comparison,performance

In [ ]:
def finish_food_report(root, artifact, config, trained, diagnostics, probes, comparison, protocol):
    artifact = Path(artifact)
    def formatted(value, digits=3):
        return "UNKNOWN" if value is None else f"{value:.{digits}f}"
    wanted = int(config["training"]["updates"])
    full = all(v["resume"]["final_step"] == wanted and v["resume"]["actual_image_exposures"] == wanted * config["effective_batch_size"] for v in trained.values())
    complete_probes = all(p["summary"]["completed_epochs"] == config["probe"]["epochs"] for p in probes.values())
    final = [r for r in diagnostics if r["step"] == wanted]
    if len(final) != 2: raise RuntimeError("Missing final representation diagnostics")
    degenerate = any(r["zero_variance_flag"] or r["very_low_rank_flag"] for r in final)
    delta = comparison["deltas"]["delta_fmm"]["correct_top1"]
    positive = full and complete_probes and delta["ci95_percentage_points"][0] > 0 and not degenerate
    comparison.update(learning_evidence="positive" if positive else "inconclusive", completed_budgets=full,
        completed_probes=complete_probes, registered_degeneracy_flags=degenerate,
        ready_for_next_stage=False, next_stage_requires_research_decision=True)
    atomic_json(artifact / "comparison_summary.json", comparison)
    performance = [v["performance"] for v in trained.values()]
    write_csv(artifact / "performance_metrics.csv", performance)
    atomic_json(artifact / "performance_metrics.json", performance)
    lines = ["# Food-101 pilot: quality and cost", "", f"Run: `{artifact.name}`. One paired training seed.",
        "", "| Representation | Development top-1 | Development top-5 |", "|---|---:|---:|"]
    for name, p in probes.items():
        m = p["summary"]["metrics"]
        lines.append(f"| {name} | {100*m['top1']:.2f}% | {100*m['top5']:.2f}% |")
    lines += ["", "| Paired top-1 contrast | Difference (pp) | 95% image-bootstrap interval (pp) |", "|---|---:|---|"]
    for name, values in comparison["deltas"].items():
        d = values["correct_top1"]
        lines.append(f"| {name} | {d['delta_percentage_points']:+.2f} | {d['ci95_percentage_points']} |")
    lines += ["", "| Model | Effective rank | Rank ceiling | Mean feature std | Mean cosine | Median full update (s) | Training wall (s) | Training peak MiB |", "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for d in final:
        p = trained[d["model_id"]]["performance"]
        seconds = p["complete_jepa_step_seconds"]["median"]
        peak = p["peak_cuda_allocated_bytes"]
        peak_mib = peak / (1024**2) if peak is not None else None
        lines.append(f"| {d['model_id']} | {d['effective_rank']:.3f} | {d['effective_rank_ceiling']} | {d['feature_std_mean']:.5f} | {d['between_image_cosine_mean']:.5f} | {formatted(seconds)} | {formatted(p['wall_seconds_including_checkpoints_and_diagnostics'],1)} | {formatted(peak_mib,1)} |")
    lines += ["", f"Registered learning evidence: **{comparison['learning_evidence']}**. Degeneration flags: **{degenerate}**.",
        "", "Interpretation: trained-minus-initial probes assess learning. The architecture contrast assesses this fixed pair only. Report actual rank and variance concentration even when advisory flags are absent.",
        "", "Limitations: one training seed; development selected probe epochs; image bootstrap excludes seed and selection uncertainty; parameter counts differ. Official test predictions remain NOT_RUN. Synthetic scaling measures compute, not visual quality. Training peak includes diagnostic allocations; allocator/context reservations are excluded. Interrupted invocation wall/peak coverage can be UNKNOWN and must not be reconstructed from reload-only measurements. Data dimension records identify native-resolution and upsampling limits.",
        "", "Next decision: if learning repeats without concerning diagnostics, consider independent seeds. If the cost is unfavorable, profile implementation before a larger budget. A near-only ablation can test the distant branch contribution. None of these follow-up runs starts automatically.",
        "", "Resume: open notebooks/02_Food101_Main.ipynb, restore this RUN_ID/config and use Run All in the main notebook. The detached worker validates all identities and resumes the latest valid checkpoint. Latest two and best checkpoints are retained under checkpoints/<run_id>/<model>/.",
        "", "Checkpoint storage is the workspace disk. Instance-deletion policy remains unverified; copy final/best checkpoints off-instance and verify hashes before recycle."]
    (artifact / "food101_results.md").write_text("\n".join(lines) + "\n")
    atomic_json(artifact / "handoff_02.json", {"run_id": artifact.name, "status": "COMPLETED",
        "learning_evidence": comparison["learning_evidence"], "heldout_status": "NOT_RUN",
        "protocol_hash": identity_hash(protocol), "source_hashes": protocol["source_hashes"],
        "report": str(artifact / "food101_results.md"), "checkpoints": {k: str(v["checkpoint_dir"]) for k, v in trained.items()},
        "no_automatic_next_stage": True})
    return comparison

def draw_food_results(artifact, diagnostics, probes, comparison, trained):
    import matplotlib.pyplot as plt
    artifact = Path(artifact)
    colors = {"vit":"#2864A0", "fmm":"#C17424"}
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
    names = list(probes)
    accuracy = [100 * probes[n]["summary"]["metrics"]["top1"] for n in names]
    bars = axes[0].bar(range(4), accuracy, color=[colors[n.split('_')[0]] for n in names])
    for bar, name in zip(bars, names):
        if name.endswith("untrained"): bar.set_hatch("//"); bar.set_alpha(.6)
    axes[0].set_xticks(range(4), [n.replace('_', '\n') for n in names]); axes[0].set_ylim(bottom=0)
    axes[0].set_ylabel("Development top-1 accuracy (%)"); axes[0].set_title("Frozen-feature classification")
    for i, v in enumerate(accuracy): axes[0].annotate(f"{v:.2f}%", (i,v), xytext=(0,4), textcoords="offset points", ha="center")
    for kind, marker in [("vit","o"),("fmm","s")]:
        rows = sorted([r for r in diagnostics if r["model_id"] == kind], key=lambda r:r["step"])
        axes[1].plot([r["step"] for r in rows], [r["effective_rank"] for r in rows], marker=marker, color=colors[kind], label=kind.upper())
        p = trained[kind]["performance"]
        axes[2].scatter(p["complete_jepa_step_seconds"]["median"], 100*probes[kind+"_trained"]["summary"]["metrics"]["top1"], marker=marker, s=75, color=colors[kind], label=kind.upper())
    axes[1].set(xlabel="Successful updates", ylabel="Raw pooled feature effective rank", title="Representation concentration")
    axes[1].set_ylim(bottom=0); axes[1].legend()
    axes[2].set(xlabel="Median full training update (seconds)", ylabel="Development top-1 accuracy (%)", title="Observed quality and training cost")
    axes[2].set_xlim(left=0); axes[2].set_ylim(bottom=0); axes[2].legend()
    for ax in axes: ax.spines[['top','right']].set_visible(False); ax.grid(axis='y',alpha=.18); ax.set_axisbelow(True)
    fig.suptitle("Food-101 pilot — one paired seed; development evaluation", fontsize=14)
    fig.savefig(artifact / "figures/food101_quality_cost.png", dpi=160)
    fig.savefig(artifact / "figures/food101_quality_cost.pdf")
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
    for i, (name, values) in enumerate(comparison["deltas"].items()):
        d = values["correct_top1"]; point = d["delta_percentage_points"]; low, high = d["ci95_percentage_points"]
        ax.plot([low,high], [i,i], color="#2864A0", linewidth=2); ax.plot(point,i,'o',color="#2864A0")
        ax.text(high,i,f"  {point:+.2f} [{low:+.2f}, {high:+.2f}]",va='center',fontsize=9)
    ax.set_yticks(range(3), list(comparison["deltas"])); ax.axvline(0,color='#555',linewidth=1)
    ax.set_xlabel("Top-1 difference (percentage points); 95% paired-image interval")
    ax.set_title("Conditional on these models; excludes seed and selection uncertainty")
    ax.margins(x=.4); fig.savefig(artifact / "figures/food101_paired_intervals.png",dpi=160); plt.close(fig)


In [ ]:
def check_compute_evidence(root, artifact, config):
    from fmm_jepa.food101_benchmark import _food_environment
    artifact = Path(artifact)
    report_path = artifact / "compute_benchmark.json"
    report = json.loads(report_path.read_text())
    manifest = json.loads((artifact / "compute_benchmark/manifest.json").read_text())
    if digest_file(report_path) != manifest.get("summary_sha256"):
        raise RuntimeError("Compute summary is incomplete or its hash changed; Run All again for verified benchmark recovery")
    identity = report["identity"]
    if identity != manifest["identity"] or identity_hash(identity) != report["identity_sha256"]:
        raise RuntimeError("Compute benchmark identity mismatch")
    if identity["configuration"] != config or identity["source_hashes"] != definition_hashes(root) or identity["environment"] != _food_environment(config):
        raise RuntimeError("Compute benchmark config/source/environment differs from this preflight")
    for record in manifest["records"].values():
        if digest_file(record["path"]) != record["sha256"]: raise RuntimeError("Compute case hash mismatch")
    tokens = (config["image_size"] // config["model"]["patch_size"]) ** 2
    selected = [r for r in report["rows"] if r["tokens"] == tokens]
    expected = {(kind,scope) for kind in ("vit","fmm") for scope in ("encoder_forward","full_jepa_update")}
    if len(selected) != 4 or {(r["model_id"],r["scope"]) for r in selected} != expected or any(r["status"] != "PASS" for r in selected):
        raise RuntimeError("The selected token count must have completed both encoder and JEPA cases for both models")
    return report

def check_preflight(artifact, config, protocol):
    artifact = Path(artifact)
    certificate = artifact / "preflight_certificate.json"
    stamp = json.loads((artifact / "preflight_binding.json").read_text())
    if stamp != {"config_hash": identity_hash(config), "protocol_hash": identity_hash(protocol), "certificate_sha256": digest_file(certificate)}:
        raise RuntimeError("Preflight certificate does not bind to this exact config/protocol")
    value = json.loads(certificate.read_text())
    if value.get("status") != "PASS": raise RuntimeError("Actual-config resume preflight must pass before training")
    from fmm_jepa.food101_training import validate_training_manifest
    if value.get("source_hashes") != protocol["source_hashes"]:
        raise RuntimeError("Preflight source hashes do not match the registered protocol")
    for kind in ("vit", "fmm"):
        record = value["models"][kind]
        for prefix in ("reference", "resumed"):
            directory = Path(record[prefix + "_directory"])
            manifest = validate_training_manifest(directory)
            if manifest["successful_updates"] != 4 or len(manifest["checkpoints"]) < 2:
                raise RuntimeError("Preflight recovery states no longer satisfy the certificate")
            if digest_file(directory / manifest["latest"]) != record[prefix + "_latest_sha256"]:
                raise RuntimeError("Preflight latest checkpoint changed after verification")
    return value

def run_food_pilot(root, run_id):
    root = Path(root).resolve(); artifact = artifact_path(root, run_id)
    if not os.environ.get("TMUX"): raise RuntimeError("Long training must run in detached tmux; use Run All in the main notebook")
    config = json.loads((artifact / "resolved_config.json").read_text())
    torch.set_num_threads(config["cpu_threads"]); torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    with run_writer(artifact):
        try:
            bundle = prepare_evidence(root, artifact, config, allow_download=False)
            models, initialization, protocol = register_food_protocol(root, artifact, config, bundle)
            check_preflight(artifact, config, protocol)
            set_status(artifact, "train_pair", "RUNNING")
            trained, diagnostics = train_pair(root, artifact, run_id, config, bundle, models, protocol)
            set_status(artifact, "four_probes", "RUNNING")
            probes, comparison, evaluation_performance = evaluate_four(root, artifact, run_id, config, bundle, trained, initialization)
            atomic_json(artifact / "evaluation_performance.json", evaluation_performance)
            set_status(artifact, "report", "RUNNING")
            comparison = finish_food_report(root, artifact, config, trained, diagnostics, probes, comparison, protocol)
            draw_food_results(artifact, diagnostics, probes, comparison, trained)
            set_status(artifact, "complete", "COMPLETED", learning_evidence=comparison["learning_evidence"])
            return comparison
        except BaseException:
            set_status(artifact, "interrupted", "FAILED", traceback=traceback.format_exc(),
                resume="same RUN_ID, unchanged config/source, Run All; latest valid checkpoints only")
            raise

def launch_food_pilot(root, run_id, config):
    import nbformat, shutil
    root = Path(root).resolve(); artifact = artifact_path(root, run_id)
    if not shutil.which("tmux"): raise RuntimeError("tmux is required for this training budget")
    if config["device"] == "cpu": raise RuntimeError("Full CPU pilot is not budgeted; select an available CUDA device")
    session = "fmm-food101-" + run_id
    registered = json.loads((artifact / "resolved_config.json").read_text())
    if registered != config: raise RuntimeError("Launch config differs from registered config")
    # Read-only existing-session lookup precedes the lock held by an active worker.
    if subprocess.run(["tmux","has-session","-t",session],capture_output=True).returncode == 0:
        return {"status":"EXISTING_SESSION", "session":session, "attach":"tmux attach -t " + session}
    with run_writer(artifact):
        registered = json.loads((artifact / "resolved_config.json").read_text())
        if registered != config: raise RuntimeError("Launch config differs from registered config")
        protocol = json.loads((artifact / "protocol.json").read_text())
        if definition_hashes(root) != protocol["source_hashes"]: raise RuntimeError("Export source changed after preflight")
        if consumer_source_hash(root) != protocol["consumer_notebook_sha256"]:
            raise RuntimeError("Main notebook changed after registration; restore it before launch")
        check_preflight(artifact, config, protocol)
        session = "fmm-food101-" + run_id
        if subprocess.run(["tmux","has-session","-t",session],capture_output=True).returncode == 0:
            return {"status":"EXISTING_SESSION", "session":session, "attach":"tmux attach -t " + session}
        status_path = artifact / "pilot_status.json"
        if status_path.exists() and json.loads(status_path.read_text()).get("status") == "COMPLETED":
            return {"status":"COMPLETED", "report":str(artifact / "food101_results.md")}
        bootstrap = json.loads((root / "notebooks/02_Food101_Main.ipynb").read_text())
        setup = next("".join(c["source"]) for c in bootstrap["cells"] if c.get("id") == "bootstrap-notebook-imports")
        code = setup + "\nfrom fmm_jepa.food101_workflow import run_food_pilot\nrun_food_pilot(ROOT, " + repr(run_id) + ")\n"
        worker = nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell(
            "# Detached Food-101 pilot worker\nGenerated by Notebook02. Resumes exact registered run. Checkpoints: checkpoints/"+run_id+"/. Expected runtime: inspect preflight certificate."), nbformat.v4.new_code_cell(code)],
            metadata={"kernelspec":{"display_name":"Python (ahr-go)","language":"python","name":"ahr-go"}})
        worker_path = artifact / "food101_worker.ipynb"
        if worker_path.exists():
            saved_worker = nbformat.read(worker_path, as_version=4)
            saved_code = [c.source for c in saved_worker.cells if c.cell_type == "code"]
            if saved_code != [code]:
                raise RuntimeError("Existing worker notebook code differs from the registered launch; restore it before resume")
        else:
            nbformat.write(worker, worker_path)
        worker_code_hash = identity_hash([code])
        command = [sys.executable,"-m","jupyter","nbconvert","--execute","--to","notebook","--inplace",
            "--ExecutePreprocessor.timeout=-1","--ExecutePreprocessor.kernel_name=ahr-go",str(worker_path)]
        shell_command = "exec " + shlex.join(command) + " > " + shlex.quote(str(artifact / "tmux_driver.log")) + " 2>&1"
        launch = {"session":session,"command":command,"attach":"tmux attach -t "+session,"detach":"Ctrl-b, then d",
            "worker":str(worker_path),"worker_code_sha256":worker_code_hash,"config_hash":identity_hash(config),"checkpoint_root":str(root / "checkpoints" / run_id)}
        atomic_json(artifact / "launch.json",launch)
        set_status(artifact,"launch","QUEUED")
    # Release the preparation writer before the detached worker acquires it.
    subprocess.run(["tmux","new-session","-d","-s",session,"-c",str(root),"bash","-c",shell_command],check=True)
    return {"status":"LAUNCHED",**launch}

def run_stage(root, run_id, config, mode, allow_download=False):
    """Explicit modes only. Preview creates no data, artifacts, process or checkpoints."""
    allowed = {"preview","benchmark","prepare","preflight","launch","review"}
    if mode not in allowed: raise ValueError(f"MODE must be one of {sorted(allowed)}")
    if mode == "preview":
        return {"status":"NOT_RUN","mode":"preview","config":config,
            "planned_artifacts":str(artifact_path(root,run_id)),"predecessor":predecessor_evidence(root,config),
            "next":"Run explicit stages after reviewing the plan; no experiment has started."}
    artifact = artifact_path(root,run_id)
    if mode == "review":
        names = ["pilot_status.json","compute_benchmark.json","preflight_certificate.json","comparison_summary.json"]
        return {n:json.loads((artifact/n).read_text()) if (artifact/n).exists() else {"status":"NOT_RUN"} for n in names}
    if mode == "launch": return launch_food_pilot(root,run_id,config)
    artifact = open_run(root,run_id,config)
    with run_writer(artifact):
        predecessor_evidence(root,config)
        if mode == "benchmark":
            from fmm_jepa.food101_benchmark import benchmark_compute
            return benchmark_compute(root,artifact,config)
        if mode == "prepare":
            bundle = prepare_evidence(root,artifact,config,allow_download)
            return {"status":"PREPARED","selected_sizes":bundle["split_manifest"]["selected_sizes"],
                "dataset_fingerprint":bundle["dataset"].fingerprint,"predictive_heldout":"NOT_RUN"}
        from fmm_jepa.food101_benchmark import verify_food_resume
        if not (artifact / "compute_benchmark.json").is_file(): raise RuntimeError("The compute benchmark must finish before preflight")
        check_compute_evidence(root,artifact,config)
        bundle = prepare_evidence(root,artifact,config,allow_download=False)
        models, initialization, protocol = register_food_protocol(root,artifact,config,bundle)
        del models
        certificate = verify_food_resume(root,artifact,config,bundle,protocol)
        if certificate.get("status") != "PASS": raise RuntimeError("Resume preflight did not pass")
        atomic_json(artifact / "preflight_binding.json", {"config_hash":identity_hash(config),
            "protocol_hash":identity_hash(protocol),"certificate_sha256":digest_file(artifact/"preflight_certificate.json")})
        set_status(artifact,"preflight","READY_TO_LAUNCH")
        return certificate


In [ ]:
def _food_session_exists(session):
    return subprocess.run(["tmux", "has-session", "-t", "=" + session], capture_output=True).returncode == 0

def run_all_context(root, run_id, config):
    """Validate existing state before deciding whether Run All prepares or reconnects."""
    import shutil
    root = Path(root).resolve()
    artifact = artifact_path(root, run_id)
    if not shutil.which("tmux"):
        raise RuntimeError("tmux is required for detached long training")
    if not str(config["device"]).startswith("cuda"):
        raise RuntimeError("This full pilot requires a CUDA device; no automatic reduced CPU experiment is substituted")
    predecessor = predecessor_evidence(root, config)
    session = "fmm-food101-" + run_id
    existing = _food_session_exists(session)
    config_path = artifact / "resolved_config.json"
    if config_path.exists() and json.loads(config_path.read_text()) != config:
        raise RuntimeError("Existing RUN_ID has a different config; restore it or choose a new RUN_ID")
    status_path = artifact / "pilot_status.json"
    status = json.loads(status_path.read_text()) if status_path.exists() else {"status":"NOT_STARTED"}
    bound = (artifact / "preflight_binding.json").exists()
    if existing or status.get("status") == "COMPLETED" or bound:
        if not config_path.is_file() or not bound:
            raise RuntimeError("Existing worker or completed status lacks registered preflight/config evidence")
        protocol = json.loads((artifact / "protocol.json").read_text())
        if protocol["config"] != config or protocol["source_hashes"] != definition_hashes(root):
            raise RuntimeError("Config or source changed after registration; resume refused")
        if protocol["consumer_notebook_sha256"] != consumer_source_hash(root):
            raise RuntimeError("Notebook code changed after registration; restore the registered source")
        check_compute_evidence(root, artifact, config)
        check_preflight(artifact, config, protocol)
        if existing:
            launch = json.loads((artifact / "launch.json").read_text())
            if launch["session"] != session or launch["config_hash"] != identity_hash(config):
                raise RuntimeError("Existing tmux session does not match this run")
        return {"skip_preparation":True, "session":session, "session_active":existing,
            "completed":status.get("status") == "COMPLETED", "predecessor":predecessor}
    return {"skip_preparation":False, "session":session, "session_active":False,
        "completed":False, "predecessor":predecessor}

def wait_for_food_pilot(root, run_id, config, poll_seconds=10):
    """Show live saved progress, surface worker failures, then return completed results.

    Interrupting this monitor does not stop the detached training process.
    """
    from IPython.display import display, Markdown
    artifact = artifact_path(root, run_id)
    session = "fmm-food101-" + run_id
    if poll_seconds < 1 or poll_seconds > 60:
        raise ValueError("Monitor interval must be between1 and60 seconds")
    handle = display(Markdown("**Food-101 pilot:** connecting to the detached worker…"), display_id=True)
    missing_since = None
    started = time.monotonic()
    try:
        while True:
            path = artifact / "pilot_status.json"
            status = json.loads(path.read_text()) if path.exists() else {"status":"QUEUED", "stage":"starting worker"}
            state = status.get("status", "UNKNOWN")
            lines = [f"**Food-101 pilot — {state}**", "", f"Stage: `{status.get('stage', 'starting worker')}`. Monitoring for {(time.monotonic()-started)/60:.1f} minutes.", ""]
            for kind in ("vit", "fmm"):
                manifest_path = Path(root) / "checkpoints" / run_id / kind / "manifest.json"
                if manifest_path.exists():
                    manifest = json.loads(manifest_path.read_text())
                    step = manifest["successful_updates"]
                    lines.append(f"- {kind.upper()}: **{step:,} / {config['training']['updates']:,}** saved updates; {manifest['image_exposures']:,} image exposures.")
                else:
                    lines.append(f"- {kind.upper()}: queued.")
            for model_id in ("vit_untrained", "vit_trained", "fmm_untrained", "fmm_trained"):
                probe_path = Path(root) / "checkpoints" / run_id / "probes" / model_id / "probe_manifest.json"
                if probe_path.exists():
                    probe = json.loads(probe_path.read_text())
                    lines.append(f"- Probe {model_id}: **{probe['latest']['completed_epochs']} / {config['probe']['epochs']}** epochs.")
            lines += ["", f"Training continues if this notebook disconnects. Attach: `tmux attach -t {session}`."]
            handle.update(Markdown("\n".join(lines)))
            if state == "COMPLETED":
                return run_stage(root, run_id, config, "review")
            if state in ("FAILED", "CANCELLED", "TERMINATED"):
                detail = status.get("traceback", status.get("error", "Inspect the worker log."))
                raise RuntimeError(f"Food-101 worker {state}. Latest valid checkpoints retained.\n{detail}\nLog: {artifact / 'tmux_driver.log'}")
            if _food_session_exists(session):
                missing_since = None
            else:
                missing_since = time.monotonic() if missing_since is None else missing_since
                if time.monotonic() - missing_since >= 30:
                    log = artifact / "tmux_driver.log"
                    tail = log.read_text(errors="replace")[-4000:] if log.exists() else "Worker log missing."
                    raise RuntimeError(f"Detached worker exited without a completed result. Saved status: {state}.\n{tail}\nFix the reported failure, then Run All to resume the same RUN_ID.")
            time.sleep(poll_seconds)
    except KeyboardInterrupt:
        print(f"Stopped monitoring. Training remains in tmux session {session}. Run All again to reconnect.", flush=True)
        raise
